# Random Forests & Boosting
**Chapter 7 — Hands-On ML**

Covers:
- Random Forest Classification & Feature Importance
- Pixel Importance on MNIST
- AdaBoost
- Gradient Boosting (manual + sklearn + early stopping)

Dataset: `make_moons`, Iris, MNIST.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_iris, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import warnings
warnings.filterwarnings('ignore')

## 1. Random Forest Classifier

Builds many decision trees on random feature subsets and bootstrap samples. Final prediction = majority vote. Reduces variance vs single tree.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)
acc = rnd_clf.score(X_test, y_test)
print(f'Accuracy: {acc:.4f}')

## 2. Feature Importance — Iris

Random Forests provide built-in feature importance: each feature's mean impurity decrease across all trees.

In [ ]:
iris = load_iris()
rnd_clf_iris = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)
rnd_clf_iris.fit(iris['data'], iris['target'])

print('Feature importances:')
for name, score in sorted(zip(iris.feature_names, rnd_clf_iris.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:<30} {score:.4f}')

## 3. Pixel Importance — MNIST

Visualizes which pixels matter most for digit classification. Center pixels are most informative.

In [ ]:
X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False, parser='auto')

rnd_clf_mnist = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rnd_clf_mnist.fit(X_mnist, y_mnist)

heatmap = rnd_clf_mnist.feature_importances_.reshape(28, 28)
plt.figure(figsize=(6, 5))
plt.imshow(heatmap, cmap='hot')
plt.colorbar(label='Feature Importance')
plt.title('MNIST — Pixel Importance Heatmap')
plt.axis('off')
plt.show()

## 4. AdaBoost

Sequentially trains weak learners — each focuses more on samples misclassified by the previous one. Final prediction = weighted majority vote.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42,
)
ada_clf.fit(X_train, y_train)
acc = ada_clf.score(X_test, y_test)
print(f'AdaBoost accuracy: {acc:.4f}')

## 5. Gradient Boosting (Manual)

Each tree fits the *residual errors* of the previous ensemble — gradient descent in function space.

In [ ]:
np.random.seed(42)
X_reg = np.random.rand(100, 1) - 0.5
y_reg = 3 * X_reg[:, 0]**2 + 0.05 * np.random.randn(100)

tree1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree1.fit(X_reg, y_reg)

y2 = y_reg - tree1.predict(X_reg)
tree2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree2.fit(X_reg, y2)

y3 = y2 - tree2.predict(X_reg)
tree3 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree3.fit(X_reg, y3)

X_new = np.array([[-0.4], [0.0], [0.5]])
y_ensemble = sum(tree.predict(X_new) for tree in (tree1, tree2, tree3))
print('Manual GBRT predictions:', y_ensemble)

## 6. Gradient Boosting — Sklearn

In [ ]:
gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt.fit(X_reg, y_reg)
print('GBRT predictions:', gbrt.predict(X_new))

## 7. Gradient Boosting — Early Stopping

Find the optimal number of trees by monitoring validation error.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, random_state=42)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, random_state=42)
gbrt.fit(X_train_r, y_train_r)

errors = [mean_squared_error(y_test_r, y_pred) for y_pred in gbrt.staged_predict(X_test_r)]
best_n = np.argmin(errors) + 1
print(f'Optimal n_estimators: {best_n}, Val MSE: {errors[best_n-1]:.6f}')

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=best_n, random_state=42)
gbrt_best.fit(X_train_r, y_train_r)
mse = mean_squared_error(y_test_r, gbrt_best.predict(X_test_r))
print(f'Best model Val MSE: {mse:.6f}')

## 8. Gradient Boosting — Incremental Early Stopping

Stop training when validation error stops improving for N consecutive rounds.

In [ ]:
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True, random_state=42)
min_val_error = float('inf')
error_going_up = 0

for n in range(1, 120):
    gbrt.n_estimators = n
    gbrt.fit(X_train_r, y_train_r)
    val_error = mean_squared_error(y_test_r, gbrt.predict(X_test_r))
    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 5:
            print(f'Early stop at n_estimators={n-5}, Val MSE={min_val_error:.6f}')
            break